# 3D Inverse SPILS-Net GNN Result Analysis (Forces $\to$ Displacements)

This notebook analyzes 3D inverse surrogate model performance for SPILS-Net with Graph Neural Network (GNN) entry and exit layers on unstructured 2D interface node distributions in 3D elastoplastic domain simulations.

**Inverse Formulation:**
- **Input ($X$):** Interface Forces/Tractions ($F_x, F_y, F_z$ for 151 interface nodes = 453 features)
- **Predicted Output ($Y$):** Interface Displacements ($u_x, u_y, u_z$ for 151 interface nodes = 453 features)
- **Internal State ($Z$):** PCA-reduced equivalent plastic strain field ($\bar{\varepsilon}^p$)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from mpl_toolkits.mplot3d import Axes3D  # Register 3D projection
from pathlib import Path

# Publication quality styling matching LaTeX standards
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "savefig.dpi": 300,
    "figure.autolayout": False,
})

In [ ]:
colors_ordered = [
    "tab:blue",
    "tab:orange",
    "tab:green",
    "tab:red",
    "tab:purple",
    "tab:brown",
    "tab:pink",
    "tab:cyan",
    "tab:olive"
]

internal_state_color = "tab:olive"

def make_plot_3d(ax, data, node_indices, dof_offset=0, n_dofs=3, linestyle='-', **kwargs):
    """
    Plots time-series data for specified node indices and DOF offset.
    
    Args:
        ax: Matplotlib axis.
        data: Array of shape [num_steps, num_nodes * n_dofs].
        node_indices: List of node indices to plot.
        dof_offset: DOF offset (0 for X, 1 for Y, 2 for Z).
        n_dofs: Number of DOFs per node (default 3 for 3D).
    """
    for i, node_idx in enumerate(node_indices):
        feature_idx = node_idx * n_dofs + dof_offset
        if feature_idx < data.shape[1]:
            ax.plot(
                data[:, feature_idx],
                linestyle=linestyle,
                color=colors_ordered[i % len(colors_ordered)],
                **kwargs
            )

## 1. Load Simulation Results & Mesh Topology

In [ ]:
# Configuration matching the simulation run
law = "plastic"
method = "spils_net_gnn"
version = "01"
freq = 1000

results_dir = Path("results_3d")
if not results_dir.exists() and Path("workspace/results_3d").exists():
    results_dir = Path("workspace/results_3d")

data_dir = Path("training_data_3d")
if not data_dir.exists() and Path("workspace/training_data_3d").exists():
    data_dir = Path("workspace/training_data_3d")

topo_path = data_dir / f"interface_topology_3d_{law}.npz"
if not topo_path.exists():
    topo_path = data_dir / "interface_topology_3d_plastic.npz"

suffix_tct = f"3d_{law}_inv_f{freq}"
suffix_sm  = f"3d_{law}_{method}_inv_v{version}_f{freq}"

# Load Full FEM Ground Truth Results (Forces as In, Displacements as Out)
tct_file_in       = results_dir / f"tct_results_in_{suffix_tct}.npy"
tct_file_out      = results_dir / f"tct_results_out_{suffix_tct}.npy"
tct_file_internal = results_dir / f"tct_results_internal_{suffix_tct}.npy"
if tct_file_in.exists() and tct_file_out.exists():
    tct_results_in  = np.load(tct_file_in)       # Ground Truth Forces
    tct_results_out = np.load(tct_file_out)      # Ground Truth Displacements
    if tct_file_internal.exists():
        tct_results_internal = np.load(tct_file_internal)
    elif (results_dir / f"tct_results_internal_3d_{law}_f{freq}.npy").exists():
        tct_results_internal = np.load(results_dir / f"tct_results_internal_3d_{law}_f{freq}.npy")
    else:
        tct_results_internal = np.zeros(tct_results_in.shape[0])
    has_true_data   = True
elif (results_dir / f"tct_results_out_3d_{law}_f{freq}.npy").exists() and (results_dir / f"tct_results_in_3d_{law}_f{freq}.npy").exists():
    # Fallback to forward TCT results: out was Forces (453 DOFs), in was [u, v, a]
    tct_results_in  = np.load(results_dir / f"tct_results_out_3d_{law}_f{freq}.npy")       # Forces
    tct_raw_u       = np.load(results_dir / f"tct_results_in_3d_{law}_f{freq}.npy")        # [u, v, a]
    tct_results_out = tct_raw_u[:, :tct_results_in.shape[-1]]                              # Displacements u
    if (results_dir / f"tct_results_internal_3d_{law}_f{freq}.npy").exists():
        tct_results_internal = np.load(results_dir / f"tct_results_internal_3d_{law}_f{freq}.npy")
    else:
        tct_results_internal = np.zeros(tct_results_in.shape[0])
    has_true_data   = True
elif (data_dir / f"training_out_3d_{law}.npz").exists() and (data_dir / f"training_in_3d_{law}_x.npz").exists():
    # Load 3D FEM Ground Truth from test simulation dataset (simulation index 4)
    test_sim_idx = 4
    tct_results_in  = np.load(data_dir / f"training_out_3d_{law}.npz")["arr_0"][test_sim_idx]    # Forces
    tct_results_out = np.load(data_dir / f"training_in_3d_{law}_x.npz")["arr_0"][test_sim_idx]  # Displacements
    tct_results_internal = np.zeros(tct_results_in.shape[0])
    has_true_data   = True
else:
    tct_results_internal = np.zeros(0)
    has_true_data   = False

# Load Inverse SM Prediction Results
sm_in_file       = results_dir / f"sm_results_in_{suffix_sm}.npy"
sm_out_file      = results_dir / f"sm_results_out_{suffix_sm}.npy"
sm_internal_file = results_dir / f"sm_results_internal_{suffix_sm}.npy"
sm_error_file    = results_dir / f"sm_results_error_{suffix_sm}.npy"

if sm_in_file.exists() and sm_out_file.exists():
    sm_results_in       = np.load(sm_in_file)          # Input Forces
    sm_results_out      = np.load(sm_out_file)         # Predicted Displacements
    sm_results_internal = np.load(sm_internal_file) if sm_internal_file.exists() else np.zeros(sm_results_in.shape[0])
    sm_results_error    = np.load(sm_error_file) if sm_error_file.exists() else np.zeros(sm_results_in.shape[0])
else:
    # Fallback to TCT ground truth data if SM evaluation not saved yet
    sm_results_in       = tct_results_in
    sm_results_out      = tct_results_out
    sm_results_internal = tct_results_internal
    sm_results_error    = np.zeros(sm_results_in.shape[0])

if not has_true_data:
    tct_results_in       = sm_results_in
    tct_results_out      = sm_results_out
    tct_results_internal = sm_results_internal

# Ensure displacement shape has only displacement DOFs (e.g. 453 instead of 1359)
n_dofs = sm_results_in.shape[-1]
if tct_results_out.shape[-1] > n_dofs:
    tct_results_out = tct_results_out[:, :n_dofs]
if sm_results_out.shape[-1] > n_dofs:
    sm_results_out = sm_results_out[:, :n_dofs]

# Trim length matching
min_len = min(tct_results_in.shape[0], sm_results_in.shape[0], tct_results_out.shape[0], sm_results_out.shape[0])
if len(tct_results_internal) > 0 and len(sm_results_internal) > 0:
    min_len = min(min_len, tct_results_internal.shape[0], sm_results_internal.shape[0])
tct_results_in      = tct_results_in[:min_len]
tct_results_out     = tct_results_out[:min_len]
tct_results_internal= tct_results_internal[:min_len]
sm_results_in       = sm_results_in[:min_len]
sm_results_out      = sm_results_out[:min_len]
sm_results_internal = sm_results_internal[:min_len]
sm_results_error    = sm_results_error[:min_len]

# Load interface mesh topology (151 nodes, 272 triangular faces)
if topo_path.exists():
    topo_data = np.load(topo_path)
    node_coords = np.asarray(topo_data["coords"], dtype=float)        # Shape: [151, 3]
    connectivity = np.asarray(topo_data["connectivity"], dtype=int)   # Shape: [272, 3]
    num_nodes = node_coords.shape[0]
else:
    num_nodes = sm_results_out.shape[1] // 3
    node_coords = None
    connectivity = None

print(f"Loaded 3D Inverse Results for {method.upper()} ({law}, v{version}, {freq}Hz):")
print(f"  Input Forces (In) shape          : {sm_results_in.shape}")
print(f"  Predicted Displacements (Out)    : {sm_results_out.shape}")
print(f"  Internal State shape             : {sm_results_internal.shape}")
print(f"  Relative Displacement Error shape: {sm_results_error.shape}")
print(f"  Has Full FEM Ground Truth        : {has_true_data}")
if has_true_data:
    print(f"  FEM Internal State shape         : {tct_results_internal.shape}")
if node_coords is not None:
    print(f"  Interface Nodes                  : {num_nodes} nodes, {connectivity.shape[0]} triangular faces")


## 2. 3D Unstructured Interface Mesh Topology Visualization

In [ ]:
sample_nodes = [0, 25, 50, 75, 100, 125, 150]

if node_coords is not None and connectivity is not None:
    fig = plt.figure(figsize=(12, 5))
    
    # Subplot 1: 3D perspective surface
    ax1 = fig.add_subplot(1, 2, 1, projection='3d')
    ax1.plot_trisurf(
        node_coords[:, 0], node_coords[:, 1], node_coords[:, 2],
        triangles=connectivity,
        cmap="viridis",
        edgecolor='gray',
        alpha=0.8,
        linewidth=0.4
    )
    ax1.scatter(
        node_coords[:, 0], node_coords[:, 1], node_coords[:, 2],
        color='crimson',
        s=12,
        zorder=5
    )
    ax1.set_title("3D Interface Surface Mesh (z = 50.0 mm)", fontsize=11)
    ax1.set_xlabel("X [mm]")
    ax1.set_ylabel("Y [mm]")
    ax1.set_zlabel("Z [mm]")
    ax1.view_init(elev=28, azim=-55)
    
    # Subplot 2: 2D planar projection (X-Y plane)
    ax2 = fig.add_subplot(1, 2, 2)
    ax2.triplot(node_coords[:, 0], node_coords[:, 1], connectivity, color='tab:blue', linewidth=0.6, alpha=0.7)
    scatter = ax2.scatter(node_coords[:, 0], node_coords[:, 1], c=np.arange(num_nodes), cmap='plasma', s=18, zorder=5)
    
    # Annotate key sample node IDs
    for n_idx in sample_nodes:
        if n_idx < num_nodes:
            ax2.annotate(
                f"{n_idx}",
                (node_coords[n_idx, 0], node_coords[n_idx, 1]),
                textcoords="offset points",
                xytext=(0, 4),
                ha='center',
                fontsize=8,
                fontweight='bold',
                color='black'
            )
            
    ax2.set_title("Planar Projection (151 Nodes, 272 Triangular Faces)", fontsize=11)
    ax2.set_xlabel("X [mm]")
    ax2.set_ylabel("Y [mm]")
    ax2.set_aspect('equal')
    ax2.grid(True, linestyle=':', alpha=0.6)
    cbar = fig.colorbar(scatter, ax=ax2, orientation='vertical', shrink=0.85)
    cbar.set_label("Node Index")
    
    fig.tight_layout()
    Path("figures").mkdir(parents=True, exist_ok=True)
    plt.savefig("figures/3d_interface_topology_inv.pdf", format="pdf", bbox_inches="tight", pad_inches=0.05)
    plt.show()
else:
    sample_nodes = [0, 10, 20, 30, 40, 50]

## 3. Overlapped 3D Response Comparison Plot (Full FEM vs Inverse SPILS-Net GNN Prediction)

This figure overlaps the **Full FEM reference ground truth** (solid, semi-transparent background track) with the **Inverse SPILS-Net GNN prediction** (dashed overlay), demonstrating agreement for **Input Forces $\to$ Predicted Displacements** across all three spatial dimensions and internal plasticity.

In [ ]:
fig, axs = plt.subplots(7, 1, figsize=(10, 14), sharex=True)

# Panel titles matching inverse formulation (Forces in, Displacements out)
axs[0].set_title(r"$(a)\quad \text{Input Interface Forces } F_x \text{ [N]}$", fontsize=10, loc="left")
axs[1].set_title(r"$(b)\quad \text{Input Interface Forces } F_y \text{ [N]}$", fontsize=10, loc="left")
axs[2].set_title(r"$(c)\quad \text{Input Interface Forces } F_z \text{ [N]}$", fontsize=10, loc="left")
axs[3].set_title(r"$(d)\quad \text{Predicted Displacements } u_x \text{ [mm]}$", fontsize=10, loc="left")
axs[4].set_title(r"$(e)\quad \text{Predicted Displacements } u_y \text{ [mm]}$", fontsize=10, loc="left")
axs[5].set_title(r"$(f)\quad \text{Predicted Displacements } u_z \text{ [mm]}$", fontsize=10, loc="left")
axs[6].set_title(r"$(g)\quad \text{Internal State } \bar{\varepsilon}^p \text{ [-]}$", fontsize=10, loc="left")
axs[6].set_xlabel("Time Step")

show_predictions = True

# --- 1. Plot GROUND TRUTH Data (Thicker, solid, semi-transparent) ---
make_plot_3d(axs[0], tct_results_in, sample_nodes, dof_offset=0, linewidth=3, alpha=0.5)
make_plot_3d(axs[1], tct_results_in, sample_nodes, dof_offset=1, linewidth=3, alpha=0.5)
make_plot_3d(axs[2], tct_results_in, sample_nodes, dof_offset=2, linewidth=3, alpha=0.5)
make_plot_3d(axs[3], tct_results_out, sample_nodes, dof_offset=0, linewidth=3, alpha=0.5)
make_plot_3d(axs[4], tct_results_out, sample_nodes, dof_offset=1, linewidth=3, alpha=0.5)
make_plot_3d(axs[5], tct_results_out, sample_nodes, dof_offset=2, linewidth=3, alpha=0.5)
axs[6].plot(tct_results_internal, linewidth=3, alpha=0.5, color=internal_state_color)

# --- 2. Plot PREDICTED Data (Thinner, dashed overlay) ---
if show_predictions:
    make_plot_3d(axs[0], sm_results_in, sample_nodes, dof_offset=0, linestyle='--', linewidth=1.5)
    make_plot_3d(axs[1], sm_results_in, sample_nodes, dof_offset=1, linestyle='--', linewidth=1.5)
    make_plot_3d(axs[2], sm_results_in, sample_nodes, dof_offset=2, linestyle='--', linewidth=1.5)
    make_plot_3d(axs[3], sm_results_out, sample_nodes, dof_offset=0, linestyle='--', linewidth=1.5)
    make_plot_3d(axs[4], sm_results_out, sample_nodes, dof_offset=1, linestyle='--', linewidth=1.5)
    make_plot_3d(axs[5], sm_results_out, sample_nodes, dof_offset=2, linestyle='--', linewidth=1.5)
    axs[6].plot(sm_results_internal, linestyle='--', linewidth=1.5, color=internal_state_color)

# Grid formatting
for ax in axs.flat:
    ax.grid(True, linestyle=':', alpha=0.7)

# Legend 1: Node Location Handles
location_handles = [
    mlines.Line2D([], [], color=colors_ordered[i % len(colors_ordered)], label=f"Node {idx}")
    for i, idx in enumerate(sample_nodes)
]

# Legend 2: Model Output Style Handles (Full FEM vs Inverse SPILS-Net GNN Prediction)
style_handles = [
    mlines.Line2D([], [], color="k", label="Full FEM", linewidth=3, alpha=0.5),
    mlines.Line2D([], [], color="k", label="Inverse SPILS-Net GNN", linestyle="--", linewidth=1.5),
]

# Location legend top-left
fig.legend(
    handles=location_handles,
    loc="upper left",
    title="Interface Node",
    ncol=len(sample_nodes),
    bbox_to_anchor=(0.02, 0.99)
)

# Model style legend top-right
fig.legend(
    handles=style_handles,
    loc="upper right",
    title="Model Output",
    ncol=2,
    bbox_to_anchor=(0.98, 0.99)
)

fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.95))
plt.savefig(f"figures/06_{method}_3d_inv_results_{freq}.pdf", format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.show()

## 4. Overlapped Displacements Zoom Plot (Near-look at Cyclic Loading Steps)

Detailed zoom examining how accurately the inverse surrogate model tracks cyclic displacement peaks and reversals under dynamic force loading.

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 7), sharex=True)

zoom_start, zoom_end = 200, 600
t_slice = slice(zoom_start, min(zoom_end, sm_results_out.shape[0]))
time_indices = np.arange(zoom_start, min(zoom_end, sm_results_out.shape[0]))

axs[0].set_title(r"$(a)\quad \text{Zoomed Displacement } u_z \text{ [mm]}$", fontsize=10, loc="left")
axs[1].set_title(r"$(b)\quad \text{Zoomed Displacement } u_x \text{ [mm]}$", fontsize=10, loc="left")
axs[2].set_title(r"$(c)\quad \text{Zoomed Displacement } u_y \text{ [mm]}$", fontsize=10, loc="left")
axs[2].set_xlabel("Time Step")

for i, node_idx in enumerate(sample_nodes):
    col = colors_ordered[i % len(colors_ordered)]
    # Z-Displacement (dof_offset = 2)
    feat_z = node_idx * 3 + 2
    if feat_z < tct_results_out.shape[1]:
        axs[0].plot(time_indices, tct_results_out[t_slice, feat_z], color=col, linewidth=3, alpha=0.5)
        axs[0].plot(time_indices, sm_results_out[t_slice, feat_z], color=col, linestyle='--', linewidth=1.5)
    
    # X-Displacement (dof_offset = 0)
    feat_x = node_idx * 3 + 0
    if feat_x < tct_results_out.shape[1]:
        axs[1].plot(time_indices, tct_results_out[t_slice, feat_x], color=col, linewidth=3, alpha=0.5)
        axs[1].plot(time_indices, sm_results_out[t_slice, feat_x], color=col, linestyle='--', linewidth=1.5)
        
    # Y-Displacement (dof_offset = 1)
    feat_y = node_idx * 3 + 1
    if feat_y < tct_results_out.shape[1]:
        axs[2].plot(time_indices, tct_results_out[t_slice, feat_y], color=col, linewidth=3, alpha=0.5)
        axs[2].plot(time_indices, sm_results_out[t_slice, feat_y], color=col, linestyle='--', linewidth=1.5)

for ax in axs:
    ax.grid(True, linestyle=':', alpha=0.7)

fig.tight_layout()
plt.savefig(f"figures/06_{method}_3d_disp_zoom_{freq}.pdf", format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.show()

## 5. Relative Nodal Displacement Error Evolution ($e_n$ Over Time)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

time_steps = len(sm_results_error)
ax.plot(sm_results_error, color="tab:blue", label=f"Inverse SPILS-Net GNN ({law.capitalize()})", linewidth=1.5)

ax.set_xlabel("Time Step")
ax.set_ylabel(r"Relative Nodal Displacement Error $e_n$")
ax.set_xlim(0, time_steps)
ax.set_ylim(0, max(np.max(sm_results_error) * 1.1, 0.01))
ax.grid(True, linestyle=':', alpha=0.6)
ax.legend(loc="upper left", frameon=True)

fig.tight_layout()
plt.savefig(f"figures/06_3d_{law}_inv_error_evolution_{freq}.pdf", format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.show()

## 6. Summary Performance Statistics

In [ ]:
mean_error  = np.mean(sm_results_error)
max_error   = np.max(sm_results_error)
final_error = sm_results_error[-1] if len(sm_results_error) > 0 else 0.0

# Global Relative Displacement Error calculation (matching create_predictor_3d_inv.py evaluate_model)
if has_true_data:
    global_err = np.linalg.norm(tct_results_out - sm_results_out) / max(np.linalg.norm(tct_results_out), 1e-10)
else:
    global_err = None

print("=" * 60)
print(f"  3D Inverse SPILS-Net GNN Performance Summary ({law.capitalize()}, {freq}Hz)")
print("=" * 60)
if global_err is not None:
    print(f"  Global Relative Error (u)   : {global_err * 100:.4f}%")
print(f"  Mean Relative Error (e_n)   : {mean_error * 100:.4f}%")
print(f"  Max Relative Error (e_n)    : {max_error * 100:.4f}%")
print(f"  Final Step Error (e_n)      : {final_error * 100:.4f}%")
print(f"  Total Simulation Time Steps : {time_steps}")
print("=" * 60)
